In [62]:
import opendatasets as od 
od.download("https://www.kaggle.com/datasets/algozee/ai-generated-vs-human-written-text-dataset")

Skipping, found downloaded files in ".\ai-generated-vs-human-written-text-dataset" (use force=True to force download)


In [63]:
import pandas as pd
import numpy as np 
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import nltk
from nltk.tokenize import word_tokenize

In [64]:
df = pd.read_csv("ai-generated-vs-human-written-text-dataset/AuthentiText_X_2026_AI_vs_Human_Detection_1K.csv", index_col="text_id")

In [65]:
df.head()

,content_text,author_type,model_source,prompt_complexity_score,perplexity_score,burstiness_index,syntactic_variability,semantic_coherence_score,lexical_diversity_ratio,readability_grade_level,generation_confidence_score
text_id,,,,,,,,,,,
TXT_0001,learning pattern detection algorithm pattern n...,AI,Human,0.029,73.75,0.953,0.465,0.351,0.187,12.2,0.162
TXT_0002,algorithm algorithm data research network mode...,Human,Claude,0.605,43.11,0.054,0.952,0.314,0.636,9.8,0.012
TXT_0003,analysis language generation research pattern ...,Human,GPT-4,0.396,59.97,0.709,0.945,0.684,0.500,13.5,0.171
TXT_0004,data language system learning content data net...,AI,GPT-4,0.299,18.99,0.532,0.780,0.216,0.103,12.9,0.838
TXT_0005,model learning content language model generati...,AI,Human,0.867,82.45,0.478,0.602,0.420,0.198,6.4,0.022


In [66]:

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\noman\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
def preprocess(text):
   
    text = text.lower()

    text = ''.join([ch if ch.isalnum() or ch in ['.', ','] or ch.isspace() else ' ' for ch in text])
    
    tokens = word_tokenize(text)
    
    return tokens

df['tokens'] = df['content_text'].apply(preprocess)

In [68]:
df.head()

,content_text,author_type,model_source,prompt_complexity_score,perplexity_score,burstiness_index,syntactic_variability,semantic_coherence_score,lexical_diversity_ratio,readability_grade_level,generation_confidence_score,tokens
text_id,,,,,,,,,,,,
TXT_0001,learning pattern detection algorithm pattern n...,AI,Human,0.029,73.75,0.953,0.465,0.351,0.187,12.2,0.162,"[learning, pattern, detection, algorithm, patt..."
TXT_0002,algorithm algorithm data research network mode...,Human,Claude,0.605,43.11,0.054,0.952,0.314,0.636,9.8,0.012,"[algorithm, algorithm, data, research, network..."
TXT_0003,analysis language generation research pattern ...,Human,GPT-4,0.396,59.97,0.709,0.945,0.684,0.500,13.5,0.171,"[analysis, language, generation, research, pat..."
TXT_0004,data language system learning content data net...,AI,GPT-4,0.299,18.99,0.532,0.780,0.216,0.103,12.9,0.838,"[data, language, system, learning, content, da..."
TXT_0005,model learning content language model generati...,AI,Human,0.867,82.45,0.478,0.602,0.420,0.198,6.4,0.022,"[model, learning, content, language, model, ge..."


In [69]:
le = LabelEncoder()
df["author_type"] = le.fit_transform(df["author_type"])

In [70]:
ss= StandardScaler()
numeric_cols = [
    'prompt_complexity_score',
    'perplexity_score',
    'burstiness_index',
    'syntactic_variability',
    'semantic_coherence_score',
    'lexical_diversity_ratio',
    'readability_grade_level',
    'generation_confidence_score'
]

for col in numeric_cols:
    df[col] = ss.fit_transform(df[[col]])

In [71]:
#training wird2vec model using skip-gram
w2v_model = Word2Vec(sentences= df["tokens"], vector_size=100, window=5, min_count=1, workers=4, sg=1)

In [72]:
def sentence_vector(tokens, model):
    vecs= [model.wv[word] for word in tokens if word in model.wv]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

df['text_vector'] = df['tokens'].apply(lambda x: sentence_vector(x, w2v_model))
X_text = np.vstack(df['text_vector'].values)

In [73]:

X_numeric = df[numeric_cols].values
X = np.hstack([X_text, X_numeric])
y = df['author_type'].values

In [74]:
X_train, X_test, y_train, y_test = train_test_split(X, y , test_size=0.2, random_state=42)

In [75]:
model = LogisticRegression(max_iter=1000, penalty='l2', C=1.0, solver='lbfgs')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred))
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred))

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.14      0.22        97
           1       0.52      0.86      0.65       103

    accuracy                           0.52       200
   macro avg       0.51      0.50      0.44       200
weighted avg       0.51      0.52      0.44       200

Logistic Regression Accuracy: 0.515


c:\Users\noman\anaconda3\envs\mlai\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
